# Notebook 3: XGBoost Model Training

**Purpose**: Train XGBoost classifier on extracted CNN features.

This notebook:
1. Loads cached features from Notebook 2
2. Trains XGBoost model on training features
3. Evaluates on test set
4. Reports baseline performance metrics

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

print(f"XGBoost version: {xgb.__version__}")

## Step 1: Load Cached Features

In [ ]:
features_dir = Path("../embeddings")

# Load training features
train_data = np.load(features_dir / "train_features.npz", allow_pickle=True)
X_train_full = train_data['features']
y_train_full = train_data['labels']

print(f"Training features shape: {X_train_full.shape}")
print(f"Training labels shape: {y_train_full.shape}")
print(f"Class distribution: {np.bincount(y_train_full)}")

# Load test features
test_data = np.load(features_dir / "test_features.npz", allow_pickle=True)
X_test = test_data['features']
y_test = test_data['labels']

print(f"\nTest features shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")
print(f"Test class distribution: {np.bincount(y_test)}")

## Step 2: Train/Validation Split

In [ ]:
# Split training data into train and validation
# Validation set is used for hyperparameter tuning and monitoring
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## Step 3: Train XGBoost Classifier

In [ ]:
# Initialize XGBoost with optimal hyperparameters
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc',
    use_label_encoder=False,
    tree_method='hist'  # Faster training
)

print("Training XGBoost model...")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=10
)

print("Training complete")

## Step 4: Evaluate on Validation and Test Sets

In [ ]:
# Predictions
y_val_pred = xgb_model.predict(X_val)
y_val_pred_proba = xgb_model.predict_proba(X_val)[:, 1]

y_test_pred = xgb_model.predict(X_test)
y_test_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

print("=" * 50)
print("VALIDATION SET METRICS")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Precision: {precision_score(y_val, y_val_pred):.4f}")
print(f"Recall:    {recall_score(y_val, y_val_pred):.4f}")
print(f"F1-Score:  {f1_score(y_val, y_val_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, y_val_pred_proba):.4f}")

print("\n" + "=" * 50)
print("TEST SET METRICS (FINAL EVALUATION)")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_test_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_test_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_test_pred_proba):.4f}")

print("\n" + "=" * 50)
print("CLASSIFICATION REPORT (TEST SET)")
print("=" * 50)
print(classification_report(y_test, y_test_pred, target_names=['NORMAL', 'PNEUMONIA']))

## Step 5: Confusion Matrix Visualization

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Validation confusion matrix
cm_val = confusion_matrix(y_val, y_val_pred)
sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Validation Set Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')
axes[0].set_xticklabels(['NORMAL', 'PNEUMONIA'])
axes[0].set_yticklabels(['NORMAL', 'PNEUMONIA'])

# Test confusion matrix
cm_test = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('Test Set Confusion Matrix')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')
axes[1].set_xticklabels(['NORMAL', 'PNEUMONIA'])
axes[1].set_yticklabels(['NORMAL', 'PNEUMONIA'])

plt.tight_layout()
plt.show()

## Step 6: Feature Importance (XGBoost Built-in)

In [ ]:
# Get feature importances
importance_dict = xgb_model.get_booster().get_score(importance_type='weight')

# Plot top 20 important features
sorted_features = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)[:20]
features, importances = zip(*sorted_features)

plt.figure(figsize=(10, 6))
plt.barh(range(len(features)), importances)
plt.yticks(range(len(features)), features)
plt.xlabel('Feature Importance (Number of Times Used)')
plt.title('Top 20 Most Important Features in XGBoost')
plt.tight_layout()
plt.show()

print(f"Total features: {len(importance_dict)}")
print(f"Top 5 features: {sorted_features[:5]}")

## Step 7: Save Model for Later Use

In [ ]:
# Save trained model
model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

# Save as pickle
model_path = model_dir / "xgb_model.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(xgb_model, f)

print(f"Model saved to {model_path}")

# Also save with XGBoost's native format
model_path_json = model_dir / "xgb_model.json"
xgb_model.get_booster().save_model(str(model_path_json))
print(f"Model also saved to {model_path_json}")

## Summary

XGBoost model trained and evaluated. Baseline performance established. Ready for SHAP analysis in Notebook 4.